**Section 1: Load the Data in PySpark**

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("ManufacturingAnalysis").getOrCreate()

In [4]:
from google.colab import files
uploaded=files.upload()

Saving manufacturing_production_120k.csv to manufacturing_production_120k.csv


In [5]:
df = spark.read.csv(
    "manufacturing_production_120k.csv",
    header=True,
    inferSchema=True
)

In [6]:
# 1. Display first 20 rows.
df.show(20)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

In [7]:
# 2. Display schema.
df.printSchema()

root
 |-- production_id: string (nullable = true)
 |-- production_date: date (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- operator_id: string (nullable = true)
 |-- shift: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- product_family: string (nullable = true)
 |-- units_planned: integer (nullable = true)
 |-- units_produced: integer (nullable = true)
 |-- defective_units: integer (nullable = true)
 |-- downtime_minutes: integer (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- vibration_mm_s: double (nullable = true)
 |-- quality_status: string (nullable = true)
 |-- maintenance_flag: string (nullable = true)
 |-- supervisor: string (nullable = true)



In [8]:
# 3. Display column names.
print(df.columns)

['production_id', 'production_date', 'plant_id', 'machine_id', 'operator_id', 'shift', 'product_code', 'product_family', 'units_planned', 'units_produced', 'defective_units', 'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s', 'quality_status', 'maintenance_flag', 'supervisor']


In [9]:
# 4. Count total records.
print("Total records:", df.count())

Total records: 120000


In [10]:
# 5. Display only production ID, plant, machine, product family and units produced.
df.select("production_id", "plant_id", "machine_id", "product_family", "units_produced").show()

+-------------+--------+----------+--------------+--------------+
|production_id|plant_id|machine_id|product_family|units_produced|
+-------------+--------+----------+--------------+--------------+
|    PR0000001|    PL03|     MC011|        Motors|           993|
|    PR0000002|    PL04|     MC019|   Controllers|           836|
|    PR0000003|    PL01|     MC002|        Motors|           380|
|    PR0000004|    PL03|     MC013|         Pumps|           720|
|    PR0000005|    PL04|     MC020|         Pumps|           296|
|    PR0000006|    PL01|     MC002|        Motors|           795|
|    PR0000007|    PL02|     MC007|        Valves|           955|
|    PR0000008|    PL02|     MC009|        Valves|           895|
|    PR0000009|    PL03|     MC015|   Controllers|           289|
|    PR0000010|    PL02|     MC006|   Controllers|           527|
|    PR0000011|    PL03|     MC012|         Pumps|           512|
|    PR0000012|    PL06|     MC030|   Controllers|           968|
|    PR000

In [11]:
# 6. Check the datatype of each column.
print(df.dtypes)

[('production_id', 'string'), ('production_date', 'date'), ('plant_id', 'string'), ('machine_id', 'string'), ('operator_id', 'string'), ('shift', 'string'), ('product_code', 'string'), ('product_family', 'string'), ('units_planned', 'int'), ('units_produced', 'int'), ('defective_units', 'int'), ('downtime_minutes', 'int'), ('energy_kwh', 'double'), ('temperature_c', 'double'), ('vibration_mm_s', 'double'), ('quality_status', 'string'), ('maintenance_flag', 'string'), ('supervisor', 'string')]


In [12]:
# 7. Display 10 random-looking sample records using sample() .
df.sample(withReplacement=False, fraction=0.01, seed=42).limit(10).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000256|     2026-02-17|    PL03|     MC012|     OP0477|Morning|     VLV-120|        Valves|          949|           790|             14|               0|    372.68|        58.34|          3.93|          Pass|              No|     Farah|
|    PR0000314|     

**Section 2: Creating DataFrames**

In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import lit

In [14]:
spark = SparkSession.builder.getOrCreate()

In [15]:
data = [
    ("P101", "Plant Alpha", "Hyderabad"),
    ("P102", "Plant Beta", "Chennai"),
    ("P103", "Plant Gamma", "Bengaluru")
]

In [16]:
# 1. Create it using a list of tuples.
df_tuples = spark.createDataFrame(data)
df_tuples.show()

+----+-----------+---------+
|  _1|         _2|       _3|
+----+-----------+---------+
|P101|Plant Alpha|Hyderabad|
|P102| Plant Beta|  Chennai|
|P103|Plant Gamma|Bengaluru|
+----+-----------+---------+



In [17]:
# 2. Create it with explicit column names.
df_plants = spark.createDataFrame(data, ["plant_id", "plant_name", "city"])
df_plants.show()

+--------+-----------+---------+
|plant_id| plant_name|     city|
+--------+-----------+---------+
|    P101|Plant Alpha|Hyderabad|
|    P102| Plant Beta|  Chennai|
|    P103|Plant Gamma|Bengaluru|
+--------+-----------+---------+



In [18]:
# 3. Create another version using StructType .
schema = StructType([
    StructField("plant_id", StringType(), True),
    StructField("plant_name", StringType(), True),
    StructField("city", StringType(), True)
])

df_struct = spark.createDataFrame(data, schema=schema)
df_struct.show()

+--------+-----------+---------+
|plant_id| plant_name|     city|
+--------+-----------+---------+
|    P101|Plant Alpha|Hyderabad|
|    P102| Plant Beta|  Chennai|
|    P103|Plant Gamma|Bengaluru|
+--------+-----------+---------+



In [19]:
# 4. Print the schema.
df_struct.printSchema()

root
 |-- plant_id: string (nullable = true)
 |-- plant_name: string (nullable = true)
 |-- city: string (nullable = true)



In [20]:
# 5. Add a constant column called country = India .
df_plants_country = df_struct.withColumn("country", lit("India"))
df_plants_country.show()

+--------+-----------+---------+-------+
|plant_id| plant_name|     city|country|
+--------+-----------+---------+-------+
|    P101|Plant Alpha|Hyderabad|  India|
|    P102| Plant Beta|  Chennai|  India|
|    P103|Plant Gamma|Bengaluru|  India|
+--------+-----------+---------+-------+



**Section 3: Pandas DataFrame**

In [21]:
import pandas as pd

pdf = pd.read_csv("manufacturing_production_120k.csv")

In [22]:
# 1. Display first 5 rows.
print(pdf.head(5))

  production_id production_date plant_id machine_id operator_id    shift  \
0     PR0000001      2026-08-07     PL03      MC011      OP0275  Evening   
1     PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2     PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
3     PR0000004      2026-01-15     PL03      MC013      OP0076  Morning   
4     PR0000005      2026-09-21     PL04      MC020      OP0501  Morning   

  product_code product_family  units_planned  units_produced  defective_units  \
0      MTR-100         Motors           1060             993               30   
1      CTL-330    Controllers            809             836                0   
2      MTR-300         Motors            442             380               17   
3      PMP-210          Pumps            787             720                7   
4      PMP-110          Pumps            344             296                7   

   downtime_minutes  energy_kwh  temperature_c  vibratio

In [23]:
# 2. Display last 5 rows.
print(pdf.tail(5))

       production_id production_date plant_id machine_id operator_id    shift  \
119995     PR0119996      2026-07-13     PL05      MC022      OP0193  Evening   
119996     PR0119997      2026-04-26     PL05      MC024      OP0124  Evening   
119997     PR0119998      2026-02-01     PL06      MC027      OP0051  Evening   
119998     PR0119999      2026-01-17     PL01      MC001      OP0452  Evening   
119999     PR0120000      2026-08-16     PL02      MC007      OP0420  Morning   

       product_code product_family  units_planned  units_produced  \
119995      MTR-200         Motors            435             399   
119996      CTL-130    Controllers            813             777   
119997      VLV-120         Valves            650             597   
119998      MTR-300         Motors            887             899   
119999      MTR-200         Motors            547             477   

        defective_units  downtime_minutes  energy_kwh  temperature_c  \
119995                1   

In [24]:
# 3. Display shape.
print("Shape:", pdf.shape)

Shape: (120000, 18)


In [25]:
# 4. Display column names.
print(pdf.columns.tolist())

['production_id', 'production_date', 'plant_id', 'machine_id', 'operator_id', 'shift', 'product_code', 'product_family', 'units_planned', 'units_produced', 'defective_units', 'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s', 'quality_status', 'maintenance_flag', 'supervisor']


In [26]:
# 5. Display datatypes.
print(pdf.dtypes)

production_id        object
production_date      object
plant_id             object
machine_id           object
operator_id          object
shift                object
product_code         object
product_family       object
units_planned         int64
units_produced        int64
defective_units       int64
downtime_minutes    float64
energy_kwh          float64
temperature_c       float64
vibration_mm_s      float64
quality_status       object
maintenance_flag     object
supervisor           object
dtype: object


In [27]:
# 6. Select plant_id , machine_id , units_produced .
print(pdf[["plant_id", "machine_id", "units_produced"]])

       plant_id machine_id  units_produced
0          PL03      MC011             993
1          PL04      MC019             836
2          PL01      MC002             380
3          PL03      MC013             720
4          PL04      MC020             296
...         ...        ...             ...
119995     PL05      MC022             399
119996     PL05      MC024             777
119997     PL06      MC027             597
119998     PL01      MC001             899
119999     PL02      MC007             477

[120000 rows x 3 columns]


In [28]:
# 7. Rename units_produced to production_output .
pdf_renamed = pdf.rename(columns={"units_produced": "production_output"})
print(pdf_renamed.head())

  production_id production_date plant_id machine_id operator_id    shift  \
0     PR0000001      2026-08-07     PL03      MC011      OP0275  Evening   
1     PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2     PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
3     PR0000004      2026-01-15     PL03      MC013      OP0076  Morning   
4     PR0000005      2026-09-21     PL04      MC020      OP0501  Morning   

  product_code product_family  units_planned  production_output  \
0      MTR-100         Motors           1060                993   
1      CTL-330    Controllers            809                836   
2      MTR-300         Motors            442                380   
3      PMP-210          Pumps            787                720   
4      PMP-110          Pumps            344                296   

   defective_units  downtime_minutes  energy_kwh  temperature_c  \
0               30              34.0      388.11          61.40   
1     

In [29]:
# 8. Filter rows where quality_status == "Reject" .
print(pdf[pdf["quality_status"] == "Reject"])

       production_id production_date plant_id machine_id operator_id    shift  \
1          PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2          PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
7          PR0000008      2026-08-17     PL02      MC009      OP0106    Night   
9          PR0000010      2026-02-13     PL02      MC006      OP0593  Evening   
21         PR0000022      2026-03-03     PL02      MC009      OP0294  Evening   
...              ...             ...      ...        ...         ...      ...   
119979     PR0119980      2026-07-21     PL02      MC006      OP0591  Morning   
119988     PR0119989      2026-01-29     PL01      MC003      OP0021    Night   
119989     PR0119990      2026-07-20     PL06      MC027      OP0217  Evening   
119991     PR0119992      2026-04-07     PL05      MC024      OP0186  Morning   
119994     PR0119995      2026-09-29     PL03      MC013      OP0151  Evening   

       product_code product

In [31]:
# 9. Filter rows where temperature is greater than 75.
print(pdf[pdf["temperature_c"] > 75])

       production_id production_date plant_id machine_id operator_id    shift  \
14         PR0000015      2026-03-15     PL02      MC007      OP0238  Morning   
16         PR0000017      2026-08-04     PL03      MC011      OP0533  Morning   
19         PR0000020      2026-02-18     PL03      MC014      OP0563  Morning   
26         PR0000027      2026-01-03     PL05      MC023      OP0190  Morning   
54         PR0000055      2026-07-31     PL01      MC005      OP0212  Morning   
...              ...             ...      ...        ...         ...      ...   
119767     PR0119768      2026-01-29     PL02      MC007      OP0262  Evening   
119810     PR0119811      2026-07-09     PL05      MC024      OP0372  Evening   
119833     PR0119834      2026-06-22     PL04      MC020      OP0038  Morning   
119848     PR0119849      2026-02-25     PL06      MC027      OP0563  Morning   
119968     PR0119969      2026-06-05     PL05      MC022      OP0366    Night   

       product_code product

In [33]:
# 10. Filter Night-shift production from Hyderabad plant.
merged_pdf = pdf.merge(df_plants, on="plant_id", how="inner")

result = merged_pdf[(merged_pdf["shift"] == "Night") & (merged_pdf["city"] == "Hyderabad")]
print(result)

TypeError: Can only merge Series or DataFrame objects, a <class 'pyspark.sql.classic.dataframe.DataFrame'> was passed

**Section 4: Dask DataFrame**

In [36]:
import dask.dataframe as dd
import pandas as pd

ddf = dd.read_csv(
    "manufacturing_production_120k.csv",
    assume_missing=True
)

In [37]:
# 1. Display first records using head() .
print(ddf.head())

  production_id production_date plant_id machine_id operator_id    shift  \
0     PR0000001      2026-08-07     PL03      MC011      OP0275  Evening   
1     PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2     PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
3     PR0000004      2026-01-15     PL03      MC013      OP0076  Morning   
4     PR0000005      2026-09-21     PL04      MC020      OP0501  Morning   

  product_code product_family  units_planned  units_produced  defective_units  \
0      MTR-100         Motors         1060.0           993.0             30.0   
1      CTL-330    Controllers          809.0           836.0              0.0   
2      MTR-300         Motors          442.0           380.0             17.0   
3      PMP-210          Pumps          787.0           720.0              7.0   
4      PMP-110          Pumps          344.0           296.0              7.0   

   downtime_minutes  energy_kwh  temperature_c  vibratio

In [38]:
# 2. Find number of partitions.
print("Number of partitions:", ddf.npartitions)

Number of partitions: 1


In [39]:
# 3. Select plant_id , product_family , units_produced .
print(ddf[["plant_id", "product_family", "units_produced"]].head())

  plant_id product_family  units_produced
0     PL03         Motors           993.0
1     PL04    Controllers           836.0
2     PL01         Motors           380.0
3     PL03          Pumps           720.0
4     PL04          Pumps           296.0


In [40]:
# 4. Filter rejected production records.
rejected_ddf = ddf[ddf["quality_status"] == "Reject"]
print(rejected_ddf.head())

   production_id production_date plant_id machine_id operator_id    shift  \
1      PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2      PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
7      PR0000008      2026-08-17     PL02      MC009      OP0106    Night   
9      PR0000010      2026-02-13     PL02      MC006      OP0593  Evening   
21     PR0000022      2026-03-03     PL02      MC009      OP0294  Evening   

   product_code product_family  units_planned  units_produced  \
1       CTL-330    Controllers          809.0           836.0   
2       MTR-300         Motors          442.0           380.0   
7       VLV-320         Valves          862.0           895.0   
9       CTL-230    Controllers          532.0           527.0   
21      MTR-300         Motors          623.0           560.0   

    defective_units  downtime_minutes  energy_kwh  temperature_c  \
1               0.0               0.0      398.58          64.37   
2         

In [41]:
# 5. Calculate average units produced by plant.
avg_units_ddf = ddf.groupby("plant_id")["units_produced"].mean()
print(avg_units_ddf)

Dask Series Structure:
npartitions=1
    float64
        ...
Dask Name: getitem, 5 expressions
Expr=((ArrowStringConversion(frame=FromMapProjectable(7f16c1f))[['plant_id', 'units_produced']]).mean(observed=False, chunk_kwargs={'numeric_only': False}, aggregate_kwargs={'numeric_only': False}, _slice='units_produced'))['units_produced']


In [42]:
# 6. Calculate total defects by product family.
defects_ddf = ddf.groupby("product_family")["defective_units"].sum()
print(defects_ddf)

Dask Series Structure:
npartitions=1
    float64
        ...
Dask Name: sum, 4 expressions
Expr=Sum(frame=ArrowStringConversion(frame=FromMapProjectable(7f16c1f))[['product_family', 'defective_units']], observed=False, chunk_kwargs={'numeric_only': False}, aggregate_kwargs={'numeric_only': False}, _slice='defective_units')


In [43]:
# 7. Use .compute() to obtain the final result.
result_units = avg_units_ddf.compute()
result_defects = defects_ddf.compute()

print("Average units by plant:\n", result_units)
print("\nTotal defects by product family:\n", result_defects)

Average units by plant:
 plant_id
PL01    672.998304
PL02    672.781326
PL03    675.665503
PL04    673.835101
PL05    673.124994
PL06    678.006884
Name: units_produced, dtype: float64

Total defects by product family:
 product_family
Controllers    507781.0
Motors         505287.0
Pumps          507864.0
Valves         509312.0
Name: defective_units, dtype: float64


**Section 5: Manipulating DataFrames**

In [44]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit
from pyspark.sql.types import DoubleType

In [45]:
# 1. Create production_efficiency = units_produced / units_planned * 100
df1 = df.withColumn("production_efficiency", (col("units_produced") / col("units_planned")) * 100)
df1.select("units_planned", "units_produced", "production_efficiency").show(5)

+-------------+--------------+---------------------+
|units_planned|units_produced|production_efficiency|
+-------------+--------------+---------------------+
|         1060|           993|    93.67924528301886|
|          809|           836|   103.33745364647713|
|          442|           380|    85.97285067873304|
|          787|           720|     91.4866581956798|
|          344|           296|    86.04651162790698|
+-------------+--------------+---------------------+
only showing top 5 rows


In [46]:
# 2. Create good_units = units_produced - defective_units
df2 = df1.withColumn("good_units", col("units_produced") - col("defective_units"))
df2.select("units_produced", "defective_units", "good_units").show(5)

+--------------+---------------+----------+
|units_produced|defective_units|good_units|
+--------------+---------------+----------+
|           993|             30|       963|
|           836|              0|       836|
|           380|             17|       363|
|           720|              7|       713|
|           296|              7|       289|
+--------------+---------------+----------+
only showing top 5 rows


In [47]:
# 3. Add country = India
df3 = df2.withColumn("country", lit("India"))
df3.select("plant_id", "country").show(5)

+--------+-------+
|plant_id|country|
+--------+-------+
|    PL03|  India|
|    PL04|  India|
|    PL01|  India|
|    PL03|  India|
|    PL04|  India|
+--------+-------+
only showing top 5 rows


In [48]:
# 4. Rename energy_kwh to energy_consumed .
df4 = df3.withColumnRenamed("energy_kwh", "energy_consumed")
df4.printSchema()

root
 |-- production_id: string (nullable = true)
 |-- production_date: date (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- operator_id: string (nullable = true)
 |-- shift: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- product_family: string (nullable = true)
 |-- units_planned: integer (nullable = true)
 |-- units_produced: integer (nullable = true)
 |-- defective_units: integer (nullable = true)
 |-- downtime_minutes: integer (nullable = true)
 |-- energy_consumed: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- vibration_mm_s: double (nullable = true)
 |-- quality_status: string (nullable = true)
 |-- maintenance_flag: string (nullable = true)
 |-- supervisor: string (nullable = true)
 |-- production_efficiency: double (nullable = true)
 |-- good_units: integer (nullable = true)
 |-- country: string (nullable = false)



In [49]:
# 5. Convert units_produced to double.
df5 = df4.withColumn("units_produced", col("units_produced").cast(DoubleType()))
df5.printSchema()

root
 |-- production_id: string (nullable = true)
 |-- production_date: date (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- operator_id: string (nullable = true)
 |-- shift: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- product_family: string (nullable = true)
 |-- units_planned: integer (nullable = true)
 |-- units_produced: double (nullable = true)
 |-- defective_units: integer (nullable = true)
 |-- downtime_minutes: integer (nullable = true)
 |-- energy_consumed: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- vibration_mm_s: double (nullable = true)
 |-- quality_status: string (nullable = true)
 |-- maintenance_flag: string (nullable = true)
 |-- supervisor: string (nullable = true)
 |-- production_efficiency: double (nullable = true)
 |-- good_units: integer (nullable = true)
 |-- country: string (nullable = false)



In [50]:
# 6. Create defect_percentage .
df6 = df5.withColumn("defect_percentage", (col("defective_units") / col("units_produced")) * 100)
df6.select("units_produced", "defective_units", "defect_percentage").show(5)

+--------------+---------------+------------------+
|units_produced|defective_units| defect_percentage|
+--------------+---------------+------------------+
|         993.0|             30|3.0211480362537766|
|         836.0|              0|               0.0|
|         380.0|             17| 4.473684210526316|
|         720.0|              7|0.9722222222222222|
|         296.0|              7| 2.364864864864865|
+--------------+---------------+------------------+
only showing top 5 rows


In [51]:
# 7. Create energy_per_unit .
df7 = df6.withColumn("energy_per_unit", col("energy_consumed") / col("units_produced"))
df7.select("units_produced", "energy_consumed", "energy_per_unit").show(5)

+--------------+---------------+-------------------+
|units_produced|energy_consumed|    energy_per_unit|
+--------------+---------------+-------------------+
|         993.0|         388.11|0.39084592145015107|
|         836.0|         398.58| 0.4767703349282296|
|         380.0|         183.34|0.48247368421052633|
|         720.0|         217.61|0.30223611111111115|
|         296.0|         115.47| 0.3901013513513513|
+--------------+---------------+-------------------+
only showing top 5 rows


In [52]:
# 8. Select only the transformed columns.
df_transformed = df7.select(
    "production_efficiency",
    "good_units",
    "country",
    "energy_consumed",
    "units_produced",
    "defect_percentage",
    "energy_per_unit"
)
df_transformed.show(5)

+---------------------+----------+-------+---------------+--------------+------------------+-------------------+
|production_efficiency|good_units|country|energy_consumed|units_produced| defect_percentage|    energy_per_unit|
+---------------------+----------+-------+---------------+--------------+------------------+-------------------+
|    93.67924528301886|       963|  India|         388.11|         993.0|3.0211480362537766|0.39084592145015107|
|   103.33745364647713|       836|  India|         398.58|         836.0|               0.0| 0.4767703349282296|
|    85.97285067873304|       363|  India|         183.34|         380.0| 4.473684210526316|0.48247368421052633|
|     91.4866581956798|       713|  India|         217.61|         720.0|0.9722222222222222|0.30223611111111115|
|    86.04651162790698|       289|  India|         115.47|         296.0| 2.364864864864865| 0.3901013513513513|
+---------------------+----------+-------+---------------+--------------+------------------+----

**Section 6: Dropping Data**

In [53]:
# 1. Drop supervisor .
df_drop_sup = df.drop("supervisor")
print(df_drop_sup.columns)

['production_id', 'production_date', 'plant_id', 'machine_id', 'operator_id', 'shift', 'product_code', 'product_family', 'units_planned', 'units_produced', 'defective_units', 'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s', 'quality_status', 'maintenance_flag']


In [54]:
# 2. Drop supervisor and maintenance_flag .
df_drop_multiple = df.drop("supervisor", "maintenance_flag")
print(df_drop_multiple.columns)

['production_id', 'production_date', 'plant_id', 'machine_id', 'operator_id', 'shift', 'product_code', 'product_family', 'units_planned', 'units_produced', 'defective_units', 'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s', 'quality_status']


In [55]:
# 3. Remove duplicate rows.
df_no_duplicates = df.dropDuplicates()
print("Count after removing duplicates:", df_no_duplicates.count())

Count after removing duplicates: 120000


In [56]:
# 4. Keep one record per production_id .
df_unique_prod = df.dropDuplicates(["production_id"])
print("Unique production records:", df_unique_prod.count())

Unique production records: 120000


In [57]:
# 5. Remove records where operator ID is missing.
df_clean_operator = df.filter(col("operator_id").isNotNull())
print("Records with operator ID:", df_clean_operator.count())

Records with operator ID: 118476


In [60]:
# 6. Remove rows where both temperature and vibration are missing.
df_clean_sensors = df.filter(~(col("temperature_c").isNull() & col("vibration_mm_s").isNull()))
print(df_clean_sensors.count())

119963


In [61]:
# 7. Display unique product families.
df.select("product_family").distinct().show()

+--------------+
|product_family|
+--------------+
|         Pumps|
|        Valves|
|   Controllers|
|        Motors|
+--------------+



In [62]:
# 8. Display unique plant-machine combinations.
df.select("plant_id", "machine_id").distinct().show()

+--------+----------+
|plant_id|machine_id|
+--------+----------+
|    PL01|     MC001|
|    PL01|     MC004|
|    PL04|     MC019|
|    PL04|     MC017|
|    PL05|     MC022|
|    PL04|     MC018|
|    PL01|     MC003|
|    PL06|     MC029|
|    PL02|     MC007|
|    PL03|     MC013|
|    PL06|     MC030|
|    PL04|     MC020|
|    PL05|     MC021|
|    PL01|     MC005|
|    PL02|     MC010|
|    PL04|     MC016|
|    PL06|     MC027|
|    PL02|     MC006|
|    PL01|     MC002|
|    PL06|     MC028|
+--------+----------+
only showing top 20 rows


**Section 7: Sorting**

In [63]:
from pyspark.sql.functions import col

In [64]:
# 1. Sort by units produced ascending.
df.sort(col("units_produced").asc()).show(5)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0038643|     2026-08-06|    PL02|     MC009|     OP0361|Evening|     MTR-100|        Motors|          250|           205|              6|              73|     65.18|        55.99|          2.72|          Pass|              No|    Harini|
|    PR0059668|     

In [65]:
# 2. Sort units produced descending.
df.sort(col("units_produced").desc()).show(5)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0087560|     2026-06-29|    PL04|     MC018|     OP0587|  Night|     VLV-120|        Valves|         1199|          1245|             60|               0|    500.87|        58.11|          3.58|          Pass|             Yes|    Charan|
|    PR0008535|     

In [66]:
# 3. Sort highest defective units first.
df.orderBy(col("defective_units").desc()).show(5)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0051359|     2026-09-16|    PL02|     MC009|     OP0181|Morning|     CTL-230|   Controllers|         1192|          1230|            108|               8|    555.34|        71.97|          2.09|        Rework|              No|     Irfan|
|    PR0108009|     

In [67]:
# 4. Sort by plant and then production output descending.
df.orderBy(col("plant_id").asc(), col("units_produced").desc()).show(10)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0073059|     2026-04-01|    PL01|     MC003|     OP0110|Evening|     MTR-200|        Motors|         1198|          1239|             11|               0|     282.6|        50.32|          3.91|        Rework|              No|     Gopal|
|    PR0002678|     

In [68]:
# 5. Display the top 20 production runs.
df.orderBy(col("units_produced").desc()).limit(20).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0087560|     2026-06-29|    PL04|     MC018|     OP0587|  Night|     VLV-120|        Valves|         1199|          1245|             60|               0|    500.87|        58.11|          3.58|          Pass|             Yes|    Charan|
|    PR0008535|     

In [69]:
# 6. Display the 10 runs with highest downtime.
df.orderBy(col("downtime_minutes").desc()).limit(10).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0092818|     2026-02-16|    PL06|     MC027|     OP0016|Morning|     PMP-210|         Pumps|         1066|           907|              2|             281|    359.48|        54.03|           2.8|          Pass|              No|     Farah|
|    PR0045363|     

In [72]:
# 7. Find machines with the highest recorded vibration.
df.select("machine_id", "vibration_mm_s").orderBy(col("vibration_mm_s").desc()).show(10)

+----------+--------------+
|machine_id|vibration_mm_s|
+----------+--------------+
|     MC012|          8.56|
|     MC030|          8.25|
|     MC027|          7.85|
|     MC010|           7.8|
|     MC018|          7.79|
|     MC024|          7.77|
|     MC015|          7.76|
|     MC027|          7.75|
|     MC015|          7.67|
|     MC016|          7.63|
+----------+--------------+
only showing top 10 rows


**Section 8: Aggregations**

In [95]:
from pyspark.sql.functions import sum, avg, max, min, count

In [74]:
# 1. Total units planned.
df.select(sum("units_planned").alias("total_units_planned")).show()

+-------------------+
|total_units_planned|
+-------------------+
|           87094621|
+-------------------+



In [75]:
# 2. Total units produced.
df.select(sum("units_produced").alias("total_units_produced")).show()

+--------------------+
|total_units_produced|
+--------------------+
|            80927888|
+--------------------+



In [76]:
# 3. Total defective units.
df.select(sum("defective_units").alias("total_defective_units")).show()

+---------------------+
|total_defective_units|
+---------------------+
|              2030244|
+---------------------+



In [77]:
# 4. Average units produced.
df.select(avg("units_produced").alias("avg_units_produced")).show()

+------------------+
|avg_units_produced|
+------------------+
| 674.3990666666666|
+------------------+



In [78]:
# 5. Maximum production output.
df.select(max("units_produced").alias("max_production_output")).show()

+---------------------+
|max_production_output|
+---------------------+
|                 1245|
+---------------------+



In [79]:
# 6. Minimum production output.
df.select(min("units_produced").alias("min_production_output")).show()

+---------------------+
|min_production_output|
+---------------------+
|                  205|
+---------------------+



In [80]:
# 7. Average downtime.
df.select(avg("downtime_minutes").alias("avg_downtime")).show()

+-----------------+
|     avg_downtime|
+-----------------+
|33.93149474428351|
+-----------------+



In [82]:
# 8. Total energy consumed.
df.select(sum("energy_kwh").alias("total_energy_consumed")).show()

+---------------------+
|total_energy_consumed|
+---------------------+
| 2.8368014080000117E7|
+---------------------+



In [84]:
# 9. Average temperature.
df.select(avg("temperature_c").alias("avg_temperature")).show()

+------------------+
|   avg_temperature|
+------------------+
|63.021532392167266|
+------------------+



In [85]:
# 10. Average vibration.
df.select(avg("vibration_mm_s").alias("avg_vibration")).show()

+-----------------+
|    avg_vibration|
+-----------------+
|3.398737691604917|
+-----------------+



**Section 9: groupBy()**

In [88]:
# 1. Total production by plant.
df.groupBy("plant_id").agg(sum("units_produced").alias("total_production")).orderBy(col("total_production").desc()).show()

+--------+----------------+
|plant_id|total_production|
+--------+----------------+
|    PL03|        13547769|
|    PL02|        13546452|
|    PL04|        13529935|
|    PL06|        13492337|
|    PL01|        13488232|
|    PL05|        13323163|
+--------+----------------+



In [90]:
# 2. Total production by product family.
df.groupBy("product_family").agg(sum("units_produced").alias("total_production")).orderBy(col("total_production").desc()).show()

+--------------+----------------+
|product_family|total_production|
+--------------+----------------+
|        Valves|        20433082|
|         Pumps|        20208739|
|   Controllers|        20168572|
|        Motors|        20117495|
+--------------+----------------+



In [91]:
# 3. Total defects by plant.
df.groupBy("plant_id").agg(sum("defective_units").alias("total_defects")).orderBy(col("total_defects").desc()).show()

+--------+-------------+
|plant_id|total_defects|
+--------+-------------+
|    PL01|       339720|
|    PL02|       339647|
|    PL03|       339264|
|    PL06|       338993|
|    PL04|       338118|
|    PL05|       334502|
+--------+-------------+



In [92]:
# 4. Average downtime by shift.
df.groupBy("shift").agg(avg("downtime_minutes").alias("avg_downtime")).orderBy(col("avg_downtime").desc()).show()

+-------+------------------+
|  shift|      avg_downtime|
+-------+------------------+
|Evening|  34.2213017151833|
|  Night|33.953058820580935|
|Morning| 33.61973080530278|
+-------+------------------+



In [93]:
# 5. Average energy usage by product family.
df.groupBy("product_family").agg(avg(energy_col).alias("avg_energy")).orderBy(col("avg_energy").desc()).show()

+--------------+------------------+
|product_family|        avg_energy|
+--------------+------------------+
|        Motors|237.21818224597516|
|         Pumps|236.89137927582252|
|        Valves|236.34508750495314|
|   Controllers|235.15370433333288|
+--------------+------------------+



In [96]:
# 6. Production count by quality status.
df.groupBy("quality_status").agg(count("*").alias("count")).orderBy(col("count").desc()).show()

+--------------+-----+
|quality_status|count|
+--------------+-----+
|          Pass|74328|
|        Rework|22928|
|        Reject|22744|
+--------------+-----+



In [97]:
# 7. Production by plant and shift.
df.groupBy("plant_id", "shift").agg(sum("units_produced").alias("total_production")).orderBy("plant_id", "shift").show()

+--------+-------+----------------+
|plant_id|  shift|total_production|
+--------+-------+----------------+
|    PL01|Evening|         4477769|
|    PL01|Morning|         4447851|
|    PL01|  Night|         4562612|
|    PL02|Evening|         4426917|
|    PL02|Morning|         4561068|
|    PL02|  Night|         4558467|
|    PL03|Evening|         4487211|
|    PL03|Morning|         4460950|
|    PL03|  Night|         4599608|
|    PL04|Evening|         4540617|
|    PL04|Morning|         4497099|
|    PL04|  Night|         4492219|
|    PL05|Evening|         4475101|
|    PL05|Morning|         4431446|
|    PL05|  Night|         4416616|
|    PL06|Evening|         4458798|
|    PL06|Morning|         4429107|
|    PL06|  Night|         4604432|
+--------+-------+----------------+



In [99]:
# 8. Defects by plant and product family.
df.groupBy("plant_id", "product_family").agg(sum("defective_units").alias("total_defects")).orderBy("plant_id", col("total_defects").desc()).show()

+--------+--------------+-------------+
|plant_id|product_family|total_defects|
+--------+--------------+-------------+
|    PL01|         Pumps|        87103|
|    PL01|        Valves|        84874|
|    PL01|        Motors|        84174|
|    PL01|   Controllers|        83569|
|    PL02|   Controllers|        87120|
|    PL02|        Motors|        84560|
|    PL02|         Pumps|        84491|
|    PL02|        Valves|        83476|
|    PL03|        Valves|        85333|
|    PL03|        Motors|        84926|
|    PL03|         Pumps|        84803|
|    PL03|   Controllers|        84202|
|    PL04|        Valves|        87084|
|    PL04|         Pumps|        83976|
|    PL04|        Motors|        83897|
|    PL04|   Controllers|        83161|
|    PL05|        Motors|        84331|
|    PL05|         Pumps|        83607|
|    PL05|   Controllers|        83589|
|    PL05|        Valves|        82975|
+--------+--------------+-------------+
only showing top 20 rows


In [101]:
# 9. Average temperature by machine.
df.groupBy("machine_id").agg(avg("temperature_c").alias("avg_temperature")).orderBy(col("avg_temperature").desc()).show()

+----------+------------------+
|machine_id|   avg_temperature|
+----------+------------------+
|     MC019|63.266223444138944|
|     MC020| 63.22489208633087|
|     MC029| 63.13119731057668|
|     MC009|  63.1252779855396|
|     MC014|63.114309957574186|
|     MC023|63.109051746112684|
|     MC027|63.099668282725126|
|     MC022| 63.09403431245128|
|     MC015| 63.09026488395557|
|     MC001|  63.0778733146782|
|     MC017| 63.07674442190667|
|     MC011|63.060230414746606|
|     MC007|   63.037661716587|
|     MC016| 63.03639870582378|
|     MC026| 63.02672521957329|
|     MC021| 63.01464069491981|
|     MC005|62.979236446933044|
|     MC028| 62.97779683034558|
|     MC030| 62.96733774497327|
|     MC024| 62.96457946271644|
+----------+------------------+
only showing top 20 rows


In [102]:
# 10. Total output by supervisor.
df.groupBy("supervisor").agg(sum("units_produced").alias("total_output")).orderBy(col("total_output").desc()).show()

+----------+------------+
|supervisor|total_output|
+----------+------------+
|     Farah|    10276986|
|    Harini|    10215803|
|     Deepa|    10130817|
|    Bharat|    10122430|
|     Anita|    10089966|
|    Charan|    10037116|
|     Gopal|    10031531|
|     Irfan|    10023239|
+----------+------------+



In [103]:
from google.colab import files
uploaded=files.upload()

Saving plant_master.csv to plant_master.csv


In [104]:
from google.colab import files
uploaded=files.upload()


Saving machine_master.csv to machine_master.csv


**Section 10: Joining DataFrames**

In [107]:
from pyspark.sql.functions import sum, avg, col, desc
production_df = spark.read.csv("manufacturing_production_120k.csv", header=True, inferSchema=True)
plant_df = spark.read.csv("plant_master.csv", header=True, inferSchema=True)
machine_df = spark.read.csv("machine_master.csv", header=True, inferSchema=True)

In [108]:
# 1. Inner join production with plant master.
joined_plant = production_df.join(plant_df, on="plant_id", how="inner")
joined_plant.show(5)

+--------+-------------+---------------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----------+-------------+
|plant_id|production_id|production_date|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|     plant_name|     city|region|year_opened|plant_manager|
+--------+-------------+---------------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----------+-------------+
|    PL03|    PR0000001|     2026-08-07|     MC011|     OP0275|Evening|     MTR-100|    

In [109]:
# 2. Display plant name instead of only plant ID.
joined_plant.select("production_id", "plant_name", "units_produced").show(5)

+-------------+---------------+--------------+
|production_id|     plant_name|units_produced|
+-------------+---------------+--------------+
|    PR0000001|  Chennai Plant|           993|
|    PR0000002|    Noida Plant|           836|
|    PR0000003|Hyderabad Plant|           380|
|    PR0000004|  Chennai Plant|           720|
|    PR0000005|    Noida Plant|           296|
+-------------+---------------+--------------+
only showing top 5 rows


In [110]:
# 3. Join production with machine master.
joined_machine = production_df.join(machine_df, on="machine_id", how="inner")
joined_machine.show(5)

+----------+-------------+---------------+--------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+--------+------------+-----------------+--------------+
|machine_id|production_id|production_date|plant_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|plant_id|machine_type|capacity_per_hour|installed_year|
+----------+-------------+---------------+--------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+--------+------------+-----------------+--------------+
|     MC011|    PR0000001|     2026-08-07|    PL03|     OP0275|Evening|     MTR-100|        Motors| 

In [111]:
# 4. Create one DataFrame containing production + plant + machine information.
complete_df = production_df.join(plant_df, on="plant_id", how="inner") \
                           .join(machine_df, on="machine_id", how="inner")
complete_df.show(5)

+----------+--------+-------------+---------------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----------+-------------+--------+------------+-----------------+--------------+
|machine_id|plant_id|production_id|production_date|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|     plant_name|     city|region|year_opened|plant_manager|plant_id|machine_type|capacity_per_hour|installed_year|
+----------+--------+-------------+---------------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----

In [112]:
# 5. Calculate total production by plant city.
complete_df.groupBy("city").agg(sum("units_produced").alias("total_production")).orderBy(desc("total_production")).show()

+---------+----------------+
|     city|total_production|
+---------+----------------+
|  Chennai|        13547769|
|     Pune|        13546452|
|    Noida|        13529935|
|  Kolkata|        13492337|
|Hyderabad|        13488232|
|Ahmedabad|        13323163|
+---------+----------------+



In [113]:
# 6. Calculate average downtime by machine type.
complete_df.groupBy("machine_type").agg(avg("downtime_minutes").alias("avg_downtime")).orderBy(desc("avg_downtime")).show()

+------------+-----------------+
|machine_type|     avg_downtime|
+------------+-----------------+
|     Casting| 34.1660783396419|
|     Welding|34.00449382099613|
|    Assembly|33.96718284443325|
|     Testing|33.87585594724829|
|         CNC|33.64548129466162|
+------------+-----------------+



In [114]:
# 7. Calculate defects by region.
complete_df.groupBy("region").agg(sum("defective_units").alias("total_defects")).orderBy(desc("total_defects")).show()

+------+-------------+
|region|total_defects|
+------+-------------+
| South|       678984|
|  West|       674149|
|  East|       338993|
| North|       338118|
+------+-------------+



In [115]:
# 8. Find production records without matching machine master records.
production_df.join(machine_df, on="machine_id", how="left_anti").show(5)

+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|machine_id|production_id|production_date|plant_id|operator_id|shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+



In [116]:
# 9. Perform a left join and inspect unmatched records.
left_joined_machine = production_df.join(machine_df, on="machine_id", how="left")
left_joined_machine.filter(col("machine_type").isNull()).show(5)

+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+--------+------------+-----------------+--------------+
|machine_id|production_id|production_date|plant_id|operator_id|shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|plant_id|machine_type|capacity_per_hour|installed_year|
+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+--------+------------+-----------------+--------------+
+----------+-------------+---------------+--------+-----------+-----+------------+--------------+---------

In [117]:
# 10. Find total energy consumption by plant name.
complete_df.groupBy("plant_name").agg(sum("energy_kwh").alias("total_energy")).orderBy(desc("total_energy")).show()

+---------------+-----------------+
|     plant_name|     total_energy|
+---------------+-----------------+
|     Pune Plant|4757984.949999988|
|    Noida Plant|4746238.580000002|
|  Chennai Plant|4742748.169999995|
|Hyderabad Plant|4725471.189999996|
|  Kolkata Plant|4723180.160000009|
|Ahmedabad Plant| 4672391.03000001|
+---------------+-----------------+



**Section 11: Updating Data**

In [118]:
from pyspark.sql.functions import col, when, coalesce, lit

spark = SparkSession.builder.getOrCreate()
df = spark.read.csv("manufacturing_production_120k.csv", header=True, inferSchema=True)

In [119]:
# 1. Change Reject to Failed .
df1 = df.withColumn(
    "quality_status",
    when(col("quality_status") == "Reject", "Failed").otherwise(col("quality_status"))
)
df1.select("quality_status").distinct().show()

+--------------+
|quality_status|
+--------------+
|        Rework|
|        Failed|
|          Pass|
+--------------+



In [120]:
# 2. Change Rework to Requires Review .
df2 = df1.withColumn(
    "quality_status",
    when(col("quality_status") == "Rework", "Requires Review").otherwise(col("quality_status"))
)
df2.select("quality_status").distinct().show()

+---------------+
| quality_status|
+---------------+
|         Failed|
|Requires Review|
|           Pass|
+---------------+



In [121]:
# 3. Increase units planned by 5% for Motors.
df3 = df2.withColumn(
    "units_planned",
    when(col("product_family") == "Motors", col("units_planned") * 1.05).otherwise(col("units_planned"))
)
df3.select("product_family", "units_planned").show(5)

+--------------+-------------+
|product_family|units_planned|
+--------------+-------------+
|        Motors|       1113.0|
|   Controllers|        809.0|
|        Motors|        464.1|
|         Pumps|        787.0|
|         Pumps|        344.0|
+--------------+-------------+
only showing top 5 rows


In [122]:
# 4. Increase energy consumption by 3% for Night shifts.
df4 = df3.withColumn(
    energy_col,
    when(col("shift") == "Night", col(energy_col) * 1.03).otherwise(col(energy_col))
)
df4.select("shift", energy_col).show(5)

+-------+----------+
|  shift|energy_kwh|
+-------+----------+
|Evening|    388.11|
|Morning|    398.58|
|Evening|    183.34|
|Morning|    217.61|
|Morning|    115.47|
+-------+----------+
only showing top 5 rows


In [123]:
# 5. Replace missing operator IDs with UNASSIGNED .
df5 = df4.withColumn(
    "operator_id",
    when(col("operator_id").isNull(), "UNASSIGNED").otherwise(col("operator_id"))
)
df5.select("operator_id").show(5)

+-----------+
|operator_id|
+-----------+
|     OP0275|
|     OP0489|
|     OP0189|
|     OP0076|
|     OP0501|
+-----------+
only showing top 5 rows


In [124]:
# 6. Replace supervisor Anita with Anita Rao .
df6 = df5.withColumn(
    "supervisor",
    when(col("supervisor") == "Anita", "Anita Rao").otherwise(col("supervisor"))
)
df6.select("supervisor").distinct().show()

+----------+
|supervisor|
+----------+
|     Farah|
|    Harini|
|     Irfan|
|    Bharat|
|    Charan|
|     Gopal|
|     Deepa|
| Anita Rao|
+----------+



In [126]:
# 7. Create a corrected temperature column.
df7 = df6.withColumn(
    "corrected_temperature",
    when(col("temperature_c") < 0, lit(None)).otherwise(col("temperature_c"))
)
df7.select("temperature_c", "corrected_temperature").show(5)

+-------------+---------------------+
|temperature_c|corrected_temperature|
+-------------+---------------------+
|         61.4|                 61.4|
|        64.37|                64.37|
|        54.13|                54.13|
|        64.47|                64.47|
|        72.59|                72.59|
+-------------+---------------------+
only showing top 5 rows


**Section 12: Null Values and na Functions**

In [127]:
# 1. Count missing operator IDs.
df.select(count(when(col("operator_id").isNull(), 1)).alias("missing_operator_ids")).show()

+--------------------+
|missing_operator_ids|
+--------------------+
|                1524|
+--------------------+



In [128]:
# 2. Count missing temperature values.
df.select(count(when(col("temperature_c").isNull(), 1)).alias("missing_temperature")).show()

+-------------------+
|missing_temperature|
+-------------------+
|               2132|
+-------------------+



In [129]:
# 3. Count missing vibration values.
df.select(count(when(col("vibration_mm_s").isNull(), 1)).alias("missing_vibration")).show()

+-----------------+
|missing_vibration|
+-----------------+
|             1788|
+-----------------+



In [130]:
# 4. Count missing downtime.
df.select(count(when(col("downtime_minutes").isNull(), 1)).alias("missing_downtime")).show()

+----------------+
|missing_downtime|
+----------------+
|            1177|
+----------------+



In [131]:
# 5. Use na.drop() .
df_dropped_all = df.na.drop()
print("Records after full na.drop():", df_dropped_all.count())

Records after full na.drop(): 113531


In [132]:
# 6. Drop only rows where temperature is missing.
df_dropped_temp = df.na.drop(subset=["temperature_c"])
print("Records after dropping missing temperature:", df_dropped_temp.count())

Records after dropping missing temperature: 117868


In [133]:
# 7. Fill missing downtime with 0 .
df_filled_downtime = df.na.fill({"downtime_minutes": 0})
df_filled_downtime.select("downtime_minutes").show(5)

+----------------+
|downtime_minutes|
+----------------+
|              34|
|               0|
|              15|
|              39|
|              49|
+----------------+
only showing top 5 rows


In [134]:
# 8. Fill missing operator ID with UNKNOWN .
df_filled_op = df.na.fill({"operator_id": "UNKNOWN"})
df_filled_op.select("operator_id").show(5)

+-----------+
|operator_id|
+-----------+
|     OP0275|
|     OP0489|
|     OP0189|
|     OP0076|
|     OP0501|
+-----------+
only showing top 5 rows


In [135]:
# 9. Fill missing temperature with the overall average temperature.
avg_temp = df.select(avg("temperature_c")).first()[0]
df_filled_temp = df.na.fill({"temperature_c": avg_temp})
df_filled_temp.select("temperature_c").show(5)

+-------------+
|temperature_c|
+-------------+
|         61.4|
|        64.37|
|        54.13|
|        64.47|
|        72.59|
+-------------+
only showing top 5 rows


**Section 13: Built-in Functions**

In [136]:
from pyspark.sql.functions import (
    upper, lower, trim, length, concat_ws,
    round, abs, coalesce, regexp_replace, substring, col, lit
)

In [137]:
# 1. Convert product family to uppercase.
df.select("product_family", upper(col("product_family")).alias("product_family_upper")).show(5)

+--------------+--------------------+
|product_family|product_family_upper|
+--------------+--------------------+
|        Motors|              MOTORS|
|   Controllers|         CONTROLLERS|
|        Motors|              MOTORS|
|         Pumps|               PUMPS|
|         Pumps|               PUMPS|
+--------------+--------------------+
only showing top 5 rows


In [138]:
# 2. Convert supervisor names to lowercase.
df.select("supervisor", lower(col("supervisor")).alias("supervisor_lower")).show(5)

+----------+----------------+
|supervisor|supervisor_lower|
+----------+----------------+
|    Charan|          charan|
|     Farah|           farah|
|     Deepa|           deepa|
|     Farah|           farah|
|     Deepa|           deepa|
+----------+----------------+
only showing top 5 rows


In [139]:
# 3. Trim operator IDs.
df.select("operator_id", trim(col("operator_id")).alias("operator_id_trimmed")).show(5)

+-----------+-------------------+
|operator_id|operator_id_trimmed|
+-----------+-------------------+
|     OP0275|             OP0275|
|     OP0489|             OP0489|
|     OP0189|             OP0189|
|     OP0076|             OP0076|
|     OP0501|             OP0501|
+-----------+-------------------+
only showing top 5 rows


In [140]:
# 4. Find length of product codes.
df.select("product_code", length(col("product_code")).alias("code_length")).show(5)

+------------+-----------+
|product_code|code_length|
+------------+-----------+
|     MTR-100|          7|
|     CTL-330|          7|
|     MTR-300|          7|
|     PMP-210|          7|
|     PMP-110|          7|
+------------+-----------+
only showing top 5 rows


In [141]:
# 5. Combine plant ID and machine ID.
df.select(concat_ws("-", col("plant_id"), col("machine_id")).alias("plant_machine")).show(5)

+-------------+
|plant_machine|
+-------------+
|   PL03-MC011|
|   PL04-MC019|
|   PL01-MC002|
|   PL03-MC013|
|   PL04-MC020|
+-------------+
only showing top 5 rows


In [142]:
# 6. Round temperature to one decimal.
df.select("temperature_c", round(col("temperature_c"), 1).alias("temperature_rounded")).show(5)

+-------------+-------------------+
|temperature_c|temperature_rounded|
+-------------+-------------------+
|         61.4|               61.4|
|        64.37|               64.4|
|        54.13|               54.1|
|        64.47|               64.5|
|        72.59|               72.6|
+-------------+-------------------+
only showing top 5 rows


In [143]:
# 7. Calculate absolute difference between planned and produced units.
df.select(
    "units_planned",
    "units_produced",
    abs(col("units_planned") - col("units_produced")).alias("abs_difference")
).show(5)

+-------------+--------------+--------------+
|units_planned|units_produced|abs_difference|
+-------------+--------------+--------------+
|         1060|           993|            67|
|          809|           836|            27|
|          442|           380|            62|
|          787|           720|            67|
|          344|           296|            48|
+-------------+--------------+--------------+
only showing top 5 rows


In [144]:
# 8. Replace characters in production ID using regexp_replace .
df.select(
    "production_id",
    regexp_replace(col("production_id"), "[^0-9]", "").alias("production_id_numeric_only")
).show(5)

+-------------+--------------------------+
|production_id|production_id_numeric_only|
+-------------+--------------------------+
|    PR0000001|                   0000001|
|    PR0000002|                   0000002|
|    PR0000003|                   0000003|
|    PR0000004|                   0000004|
|    PR0000005|                   0000005|
+-------------+--------------------------+
only showing top 5 rows


In [145]:
# 9. Extract part of product code using substring .
df.select("product_code", substring(col("product_code"), 1, 3).alias("code_prefix")).show(5)

+------------+-----------+
|product_code|code_prefix|
+------------+-----------+
|     MTR-100|        MTR|
|     CTL-330|        CTL|
|     MTR-300|        MTR|
|     PMP-210|        PMP|
|     PMP-110|        PMP|
+------------+-----------+
only showing top 5 rows


In [146]:
# 10. Use coalesce() to return temperature or a default value.
df.select(
    "temperature_c",
    coalesce(col("temperature_c"), lit(25.0)).alias("temperature_coalesced")
).show(5)

+-------------+---------------------+
|temperature_c|temperature_coalesced|
+-------------+---------------------+
|         61.4|                 61.4|
|        64.37|                64.37|
|        54.13|                54.13|
|        64.47|                64.47|
|        72.59|                72.59|
+-------------+---------------------+
only showing top 5 rows


**Section 14: when() / otherwise()**

In [147]:
df_prep = df.withColumn("efficiency_pct", (col("units_produced") / col("units_planned")) * 100)

In [148]:
# Efficiency Category
df_efficiency = df_prep.withColumn(
    "efficiency_category",
    when(col("efficiency_pct") >= 100, "Above Target")
    .when(col("efficiency_pct") >= 95, "On Target")
    .when(col("efficiency_pct") >= 85, "Below Target")
    .otherwise("Critical")
)
df_efficiency.select("efficiency_pct", "efficiency_category").show(5)

+------------------+-------------------+
|    efficiency_pct|efficiency_category|
+------------------+-------------------+
| 93.67924528301886|       Below Target|
|103.33745364647713|       Above Target|
| 85.97285067873304|       Below Target|
|  91.4866581956798|       Below Target|
| 86.04651162790698|       Below Target|
+------------------+-------------------+
only showing top 5 rows


In [149]:
# Defect Category
df_defect = df_efficiency.withColumn(
    "defect_category",
    when(col("defective_units") == 0, "Perfect")
    .when((col("defective_units") >= 1) & (col("defective_units") <= 10), "Normal")
    .when((col("defective_units") >= 11) & (col("defective_units") <= 25), "Attention")
    .otherwise("High Defect")
)
df_defect.select("defective_units", "defect_category").show(5)

+---------------+---------------+
|defective_units|defect_category|
+---------------+---------------+
|             30|    High Defect|
|              0|        Perfect|
|             17|      Attention|
|              7|         Normal|
|              7|         Normal|
+---------------+---------------+
only showing top 5 rows


In [150]:
# Machine Condition
# High temperature OR high vibration -> Inspect otherwise -> Normal
df_categorized = df_defect.withColumn(
    "machine_condition",
    when((col("temperature_c") > 80) | (col("vibration_mm_s") > 0.5), "Inspect")
    .otherwise("Normal")
)
df_categorized.select("temperature_c", "vibration_mm_s", "machine_condition").show(5)

+-------------+--------------+-----------------+
|temperature_c|vibration_mm_s|machine_condition|
+-------------+--------------+-----------------+
|         61.4|          3.88|          Inspect|
|        64.37|          2.28|          Inspect|
|        54.13|          4.66|          Inspect|
|        64.47|          NULL|           Normal|
|        72.59|          3.21|          Inspect|
+-------------+--------------+-----------------+
only showing top 5 rows


In [151]:
# Then count records in each resulting category.
print("Efficiency Categories:")
df_categorized.groupBy("efficiency_category").count().show()

print("Defect Categories:")
df_categorized.groupBy("defect_category").count().show()

print("Machine Condition Categories:")
df_categorized.groupBy("machine_condition").count().show()

Efficiency Categories:
+-------------------+-----+
|efficiency_category|count|
+-------------------+-----+
|       Above Target|21735|
|          On Target|26825|
|       Below Target|54619|
|           Critical|16821|
+-------------------+-----+

Defect Categories:
+---------------+-----+
|defect_category|count|
+---------------+-----+
|      Attention|45745|
|        Perfect|11491|
|    High Defect|28058|
|         Normal|34706|
+---------------+-----+

Machine Condition Categories:
+-----------------+------+
|machine_condition| count|
+-----------------+------+
|          Inspect|117526|
|           Normal|  2474|
+-----------------+------+



**Section 15: Complex JSON**

In [152]:
from google.colab import files
uploaded=files.upload()

Saving machine_configurations.json to machine_configurations.json


In [153]:
from pyspark.sql.functions import col, explode
machine_json = spark.read.option("multiline", "true").json("machine_configurations.json")

In [154]:
# 1. Display schema.
machine_json.printSchema()

root
 |-- capacity_per_hour: long (nullable = true)
 |-- installed_year: long (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- machine_type: string (nullable = true)
 |-- maintenance: struct (nullable = true)
 |    |-- frequency_days: long (nullable = true)
 |    |-- last_service: string (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- sensors: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- active: boolean (nullable = true)
 |    |    |-- sensor: string (nullable = true)
 |    |    |-- unit: string (nullable = true)
 |-- specifications: struct (nullable = true)
 |    |-- max_temperature_c: long (nullable = true)
 |    |-- rated_power_kw: long (nullable = true)
 |    |-- vibration_limit_mm_s: double (nullable = true)



In [155]:
# 2. Display machine ID and machine type.
machine_json.select("machine_id", "machine_type").show(5)

+----------+------------+
|machine_id|machine_type|
+----------+------------+
|     MC001|         CNC|
|     MC002|    Assembly|
|     MC003|     Casting|
|     MC004|     Welding|
|     MC005|     Testing|
+----------+------------+
only showing top 5 rows


In [156]:
# 3. Extract specifications.rated_power_kw .
machine_json.select("machine_id", col("specifications.rated_power_kw")).show(5)

+----------+--------------+
|machine_id|rated_power_kw|
+----------+--------------+
|     MC001|            22|
|     MC002|            22|
|     MC003|            55|
|     MC004|            22|
|     MC005|            30|
+----------+--------------+
only showing top 5 rows


In [158]:
# 4. Extract maximum temperature.
machine_json.select("machine_id", col("specifications.max_temperature_c")).show(5)

+----------+-----------------+
|machine_id|max_temperature_c|
+----------+-----------------+
|     MC001|               85|
|     MC002|               85|
|     MC003|               90|
|     MC004|               85|
|     MC005|               85|
+----------+-----------------+
only showing top 5 rows


In [159]:
# 5. Extract vibration limit.
machine_json.select("machine_id", col("specifications.vibration_limit_mm_s")).show(5)

+----------+--------------------+
|machine_id|vibration_limit_mm_s|
+----------+--------------------+
|     MC001|                 5.0|
|     MC002|                 5.5|
|     MC003|                 4.5|
|     MC004|                 4.5|
|     MC005|                 4.5|
+----------+--------------------+
only showing top 5 rows


In [162]:
# 6. Display maintenance frequency.
machine_json.select("machine_id", "maintenance.frequency_days").show(5)

+----------+--------------+
|machine_id|frequency_days|
+----------+--------------+
|     MC001|            60|
|     MC002|            60|
|     MC003|            45|
|     MC004|            60|
|     MC005|            30|
+----------+--------------+
only showing top 5 rows


In [163]:
# 7. Work with the sensors array.
machine_json.select("machine_id", "sensors").show(5, truncate=False)

+----------+----------------------------------------------------------------------+
|machine_id|sensors                                                               |
+----------+----------------------------------------------------------------------+
|MC001     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC002     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC003     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC004     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC005     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
+----------+----------------------------------------------------------------------+
only showing top 5 rows


In [164]:
# 8. Explode the sensors array.
exploded_sensors = machine_json.withColumn("sensor", explode(col("sensors")))
exploded_sensors.select("machine_id", "sensor").show(5, truncate=False)

+----------+-----------------------+
|machine_id|sensor                 |
+----------+-----------------------+
|MC001     |{true, temperature, C} |
|MC001     |{true, vibration, mm/s}|
|MC001     |{true, energy, kWh}    |
|MC002     |{true, temperature, C} |
|MC002     |{true, vibration, mm/s}|
+----------+-----------------------+
only showing top 5 rows


In [165]:
# 9. Display one row per machine/sensor combination.
exploded_sensors.select("machine_id", "machine_type", "sensor.*").show(10)

+----------+------------+------+-----------+----+
|machine_id|machine_type|active|     sensor|unit|
+----------+------------+------+-----------+----+
|     MC001|         CNC|  true|temperature|   C|
|     MC001|         CNC|  true|  vibration|mm/s|
|     MC001|         CNC|  true|     energy| kWh|
|     MC002|    Assembly|  true|temperature|   C|
|     MC002|    Assembly|  true|  vibration|mm/s|
|     MC002|    Assembly|  true|     energy| kWh|
|     MC003|     Casting|  true|temperature|   C|
|     MC003|     Casting|  true|  vibration|mm/s|
|     MC003|     Casting|  true|     energy| kWh|
|     MC004|     Welding|  true|temperature|   C|
+----------+------------+------+-----------+----+
only showing top 10 rows


In [167]:
# 10. Find machines having an active temperature sensor.
active_temp_machines = exploded_sensors.filter(
    (lower(col("sensor.sensor")).contains("temp")) & (col("sensor.active") == True)
).select("machine_id").distinct()

active_temp_machines.show()

+----------+
|machine_id|
+----------+
|     MC003|
|     MC018|
|     MC017|
|     MC009|
|     MC014|
|     MC007|
|     MC002|
|     MC024|
|     MC016|
|     MC026|
|     MC005|
|     MC010|
|     MC012|
|     MC023|
|     MC021|
|     MC020|
|     MC011|
|     MC013|
|     MC025|
|     MC028|
+----------+
only showing top 20 rows


In [168]:
# 11. Join machine configuration JSON with production data.
prod_with_config = production_df.join(
    machine_json.drop("plant_id"),
    on="machine_id",
    how="inner"
)
prod_with_config.show(5)

+----------+-------------+---------------+--------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+-----------------+--------------+------------+----------------+--------------------+--------------+
|machine_id|production_id|production_date|plant_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|capacity_per_hour|installed_year|machine_type|     maintenance|             sensors|specifications|
+----------+-------------+---------------+--------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+-----------------+--------------+------------+----------------+-----

**Section 16: Union and Union ALL**

In [169]:
from google.colab import files
uploaded=files.upload()

Saving manufacturing_production_extra_12k.csv to manufacturing_production_extra_12k.csv


In [170]:
extra_df = spark.read.csv("manufacturing_production_extra_12k.csv", header=True, inferSchema=True)

In [171]:
# 1. Count records in the main DataFrame.
main_count = df.count()
print("Main DataFrame record count:", main_count)

Main DataFrame record count: 120000


In [172]:
# 2. Count records in the extra DataFrame.
extra_count = extra_df.count()
print("Extra DataFrame record count:", extra_count)

Extra DataFrame record count: 12500


In [173]:
# 3. Combine using union() .
df_union = df.union(extra_df)
print("Combined count using union():", df_union.count())

Combined count using union(): 132500


In [174]:
# 4. Combine using unionAll() .
df_union_all = df.unionAll(extra_df)
print("Combined count using unionAll():", df_union_all.count())

Combined count using unionAll(): 132500


In [175]:
# 5. Compare resulting counts.
print(f"Main records: {main_count}")
print(f"Extra records: {extra_count}")
print(f"Expected sum: {main_count + extra_count}")
print(f"union() count: {df_union.count()}")
print(f"unionAll() count: {df_union_all.count()}")

Main records: 120000
Extra records: 12500
Expected sum: 132500
union() count: 132500
unionAll() count: 132500


In [176]:
# 6. Identify duplicate production_id values.
duplicates = df_union.groupBy("production_id").count().filter(col("count") > 1)
duplicates.show(10)

+-------------+-----+
|production_id|count|
+-------------+-----+
|    PR0000413|    2|
|    PR0000135|    2|
|    PR0000236|    2|
|    PR0000339|    2|
|    PR0000059|    2|
|    PR0000463|    2|
|    PR0000136|    2|
|    PR0000263|    2|
|    PR0000393|    2|
|    PR0000445|    2|
+-------------+-----+
only showing top 10 rows


In [177]:
# 7. Remove duplicates after combining.
df_deduped = df_union.dropDuplicates(["production_id"])

In [178]:
# 8. Compare unioned count before and after dropDuplicates() .
print("Count before dropDuplicates():", df_union.count())
print("Count after dropDuplicates():", df_deduped.count())

Count before dropDuplicates(): 132500
Count after dropDuplicates(): 132000


In [179]:
# 9. Try unionByName() .
df_union_name = df.unionByName(extra_df)
print("Count with unionByName():", df_union_name.count())

Count with unionByName(): 132500


In [180]:
# 10. Rearrange columns in one DataFrame and use unionByName() appropriately.
reversed_cols = extra_df.columns[::-1]
extra_df_reordered = extra_df.select(reversed_cols)

df_reordered_union = df.unionByName(extra_df_reordered)
print("Count after unionByName with reordered columns:", df_reordered_union.count())

Count after unionByName with reordered columns: 132500


**Section 17: Window Functions**

In [181]:
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    row_number, rank, dense_rank, sum, avg, lag, lead, col, desc
)
df = spark.read.csv("manufacturing_production_120k.csv", header=True, inferSchema=True)
plant_window_desc = Window.partitionBy("plant_id").orderBy(desc("units_produced"))
plant_window_unbounded = Window.partitionBy("plant_id")
machine_order_window = Window.partitionBy("machine_id").orderBy("production_date")

In [182]:
# 1. Rank production records within each plant by units produced.
df.withColumn("rank", rank().over(plant_window_desc)) \
  .select("plant_id", "production_id", "units_produced", "rank").show(5)

+--------+-------------+--------------+----+
|plant_id|production_id|units_produced|rank|
+--------+-------------+--------------+----+
|    PL01|    PR0073059|          1239|   1|
|    PL01|    PR0002678|          1238|   2|
|    PL01|    PR0005192|          1236|   3|
|    PL01|    PR0044516|          1236|   3|
|    PL01|    PR0056319|          1236|   3|
+--------+-------------+--------------+----+
only showing top 5 rows


In [183]:
# 2. Use row_number() .
df.withColumn("row_num", row_number().over(plant_window_desc)) \
  .select("plant_id", "production_id", "units_produced", "row_num").show(5)

+--------+-------------+--------------+-------+
|plant_id|production_id|units_produced|row_num|
+--------+-------------+--------------+-------+
|    PL01|    PR0073059|          1239|      1|
|    PL01|    PR0002678|          1238|      2|
|    PL01|    PR0005192|          1236|      3|
|    PL01|    PR0044516|          1236|      4|
|    PL01|    PR0056319|          1236|      5|
+--------+-------------+--------------+-------+
only showing top 5 rows


In [184]:
# 3. Use rank() .
df.withColumn("rank", rank().over(plant_window_desc)) \
  .select("plant_id", "production_id", "units_produced", "rank").show(5)

+--------+-------------+--------------+----+
|plant_id|production_id|units_produced|rank|
+--------+-------------+--------------+----+
|    PL01|    PR0073059|          1239|   1|
|    PL01|    PR0002678|          1238|   2|
|    PL01|    PR0005192|          1236|   3|
|    PL01|    PR0044516|          1236|   3|
|    PL01|    PR0056319|          1236|   3|
+--------+-------------+--------------+----+
only showing top 5 rows


In [185]:
# 4. Use dense_rank() .
df.withColumn("dense_rank", dense_rank().over(plant_window_desc)) \
  .select("plant_id", "production_id", "units_produced", "dense_rank").show(5)

+--------+-------------+--------------+----------+
|plant_id|production_id|units_produced|dense_rank|
+--------+-------------+--------------+----------+
|    PL01|    PR0073059|          1239|         1|
|    PL01|    PR0002678|          1238|         2|
|    PL01|    PR0005192|          1236|         3|
|    PL01|    PR0044516|          1236|         3|
|    PL01|    PR0056319|          1236|         3|
+--------+-------------+--------------+----------+
only showing top 5 rows


In [186]:
# 5. Find top 5 production runs per plant.
df.withColumn("rn", row_number().over(plant_window_desc)) \
  .filter(col("rn") <= 5) \
  .select("plant_id", "production_id", "units_produced", "rn").show(15)

+--------+-------------+--------------+---+
|plant_id|production_id|units_produced| rn|
+--------+-------------+--------------+---+
|    PL01|    PR0073059|          1239|  1|
|    PL01|    PR0002678|          1238|  2|
|    PL01|    PR0005192|          1236|  3|
|    PL01|    PR0044516|          1236|  4|
|    PL01|    PR0056319|          1236|  5|
|    PL02|    PR0100473|          1245|  1|
|    PL02|    PR0010732|          1243|  2|
|    PL02|    PR0001765|          1242|  3|
|    PL02|    PR0027036|          1238|  4|
|    PL02|    PR0012982|          1237|  5|
|    PL03|    PR0082579|          1243|  1|
|    PL03|    PR0030298|          1239|  2|
|    PL03|    PR0117602|          1239|  3|
|    PL03|    PR0102987|          1236|  4|
|    PL03|    PR0021534|          1235|  5|
+--------+-------------+--------------+---+
only showing top 15 rows


In [187]:
# 6. Find top 3 machines by output within each plant.
plant_machine_totals = df.groupBy("plant_id", "machine_id").agg(sum("units_produced").alias("total_output"))
machine_window = Window.partitionBy("plant_id").orderBy(desc("total_output"))

plant_machine_totals.withColumn("machine_rank", dense_rank().over(machine_window)) \
                    .filter(col("machine_rank") <= 3).show(15)

+--------+----------+------------+------------+
|plant_id|machine_id|total_output|machine_rank|
+--------+----------+------------+------------+
|    PL01|     MC004|     2722412|           1|
|    PL01|     MC002|     2713779|           2|
|    PL01|     MC001|     2691431|           3|
|    PL02|     MC009|     2760070|           1|
|    PL02|     MC007|     2713038|           2|
|    PL02|     MC008|     2705850|           3|
|    PL03|     MC012|     2750570|           1|
|    PL03|     MC015|     2739284|           2|
|    PL03|     MC014|     2735500|           3|
|    PL04|     MC016|     2750780|           1|
|    PL04|     MC019|     2735839|           2|
|    PL04|     MC017|     2726300|           3|
|    PL05|     MC024|     2737331|           1|
|    PL05|     MC023|     2718075|           2|
|    PL05|     MC022|     2635629|           3|
+--------+----------+------------+------------+
only showing top 15 rows


In [188]:
# 7. Calculate total plant production without collapsing rows.
df.withColumn("plant_total_production", sum("units_produced").over(plant_window_unbounded)) \
  .select("production_id", "plant_id", "units_produced", "plant_total_production").show(5)

+-------------+--------+--------------+----------------------+
|production_id|plant_id|units_produced|plant_total_production|
+-------------+--------+--------------+----------------------+
|    PR0000003|    PL01|           380|              13488232|
|    PR0000006|    PL01|           795|              13488232|
|    PR0000019|    PL01|           380|              13488232|
|    PR0000026|    PL01|           663|              13488232|
|    PR0000028|    PL01|          1042|              13488232|
+-------------+--------+--------------+----------------------+
only showing top 5 rows


In [189]:
# 8. Calculate average machine output using a window.
machine_window_unbounded = Window.partitionBy("machine_id")

df.withColumn("avg_machine_output", avg("units_produced").over(machine_window_unbounded)) \
  .select("production_id", "machine_id", "units_produced", "avg_machine_output").show(5)

+-------------+----------+--------------+------------------+
|production_id|machine_id|units_produced|avg_machine_output|
+-------------+----------+--------------+------------------+
|    PR0000026|     MC001|           663| 671.5147205588822|
|    PR0000057|     MC001|           325| 671.5147205588822|
|    PR0000088|     MC001|           676| 671.5147205588822|
|    PR0000135|     MC001|           375| 671.5147205588822|
|    PR0000139|     MC001|           378| 671.5147205588822|
+-------------+----------+--------------+------------------+
only showing top 5 rows


In [190]:
# 9. Calculate cumulative production by machine ordered by date.
cum_window = Window.partitionBy("machine_id").orderBy("production_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df.withColumn("cumulative_production", sum("units_produced").over(cum_window)) \
  .select("machine_id", "production_date", "units_produced", "cumulative_production").show(5)

+----------+---------------+--------------+---------------------+
|machine_id|production_date|units_produced|cumulative_production|
+----------+---------------+--------------+---------------------+
|     MC001|     2026-01-01|           437|                  437|
|     MC001|     2026-01-01|           997|                 1434|
|     MC001|     2026-01-01|           822|                 2256|
|     MC001|     2026-01-01|           817|                 3073|
|     MC001|     2026-01-01|           784|                 3857|
+----------+---------------+--------------+---------------------+
only showing top 5 rows


In [191]:
# 10. Display previous production output using lag() .
df.withColumn("prev_units_produced", lag("units_produced", 1).over(machine_order_window)) \
  .select("machine_id", "production_date", "units_produced", "prev_units_produced").show(5)

+----------+---------------+--------------+-------------------+
|machine_id|production_date|units_produced|prev_units_produced|
+----------+---------------+--------------+-------------------+
|     MC001|     2026-01-01|           437|               NULL|
|     MC001|     2026-01-01|           997|                437|
|     MC001|     2026-01-01|           822|                997|
|     MC001|     2026-01-01|           817|                822|
|     MC001|     2026-01-01|           784|                817|
+----------+---------------+--------------+-------------------+
only showing top 5 rows


In [192]:
# 11. Display next production output using lead() .
df.withColumn("next_units_produced", lead("units_produced", 1).over(machine_order_window)) \
  .select("machine_id", "production_date", "units_produced", "next_units_produced").show(5)

+----------+---------------+--------------+-------------------+
|machine_id|production_date|units_produced|next_units_produced|
+----------+---------------+--------------+-------------------+
|     MC001|     2026-01-01|           437|                997|
|     MC001|     2026-01-01|           997|                822|
|     MC001|     2026-01-01|           822|                817|
|     MC001|     2026-01-01|           817|                784|
|     MC001|     2026-01-01|           784|                472|
+----------+---------------+--------------+-------------------+
only showing top 5 rows


In [193]:
# 12. Calculate change from the previous production run.
df.withColumn("prev_units", lag("units_produced", 1).over(machine_order_window)) \
  .withColumn("output_change", col("units_produced") - col("prev_units")) \
  .select("machine_id", "production_date", "units_produced", "prev_units", "output_change").show(5)

+----------+---------------+--------------+----------+-------------+
|machine_id|production_date|units_produced|prev_units|output_change|
+----------+---------------+--------------+----------+-------------+
|     MC001|     2026-01-01|           437|      NULL|         NULL|
|     MC001|     2026-01-01|           997|       437|          560|
|     MC001|     2026-01-01|           822|       997|         -175|
|     MC001|     2026-01-01|           817|       822|           -5|
|     MC001|     2026-01-01|           784|       817|          -33|
+----------+---------------+--------------+----------+-------------+
only showing top 5 rows


In [194]:
# 13. Rank product families by total output.
family_totals = df.groupBy("product_family").agg(sum("units_produced").alias("total_output"))
all_window = Window.orderBy(desc("total_output"))

family_totals.withColumn("family_rank", rank().over(all_window)).show()

+--------------+------------+-----------+
|product_family|total_output|family_rank|
+--------------+------------+-----------+
|        Valves|    20433082|          1|
|         Pumps|    20208739|          2|
|   Controllers|    20168572|          3|
|        Motors|    20117495|          4|
+--------------+------------+-----------+



**Section 18: Date and Time Functions**

In [195]:
from pyspark.sql.functions import (
    to_date, year, month, dayofmonth, date_format,
    sum, col, desc, date_add, date_sub, datediff, current_date
)
raw_df = spark.read.csv("manufacturing_production_120k.csv", header=True, inferSchema=True)

In [196]:
# 1. Convert to DateType .
df = raw_df.withColumn("production_date", to_date(col("production_date")))
df.select("production_date").printSchema()

root
 |-- production_date: date (nullable = true)



In [197]:
# 2. Extract year.
df.select("production_date", year("production_date").alias("year")).show(5)

+---------------+----+
|production_date|year|
+---------------+----+
|     2026-08-07|2026|
|     2026-08-03|2026|
|     2026-05-20|2026|
|     2026-01-15|2026|
|     2026-09-21|2026|
+---------------+----+
only showing top 5 rows


In [198]:
# 3. Extract month.
df.select("production_date", month("production_date").alias("month")).show(5)

+---------------+-----+
|production_date|month|
+---------------+-----+
|     2026-08-07|    8|
|     2026-08-03|    8|
|     2026-05-20|    5|
|     2026-01-15|    1|
|     2026-09-21|    9|
+---------------+-----+
only showing top 5 rows


In [199]:
# 4. Extract day.
df.select("production_date", dayofmonth("production_date").alias("day")).show(5)

+---------------+---+
|production_date|day|
+---------------+---+
|     2026-08-07|  7|
|     2026-08-03|  3|
|     2026-05-20| 20|
|     2026-01-15| 15|
|     2026-09-21| 21|
+---------------+---+
only showing top 5 rows


In [200]:
# 5. Extract month name.
df.select("production_date", date_format("production_date", "MMMM").alias("month_name")).show(5)

+---------------+----------+
|production_date|month_name|
+---------------+----------+
|     2026-08-07|    August|
|     2026-08-03|    August|
|     2026-05-20|       May|
|     2026-01-15|   January|
|     2026-09-21| September|
+---------------+----------+
only showing top 5 rows


In [201]:
# 6. Extract day name.
df.select("production_date", date_format("production_date", "EEEE").alias("day_name")).show(5)

+---------------+---------+
|production_date| day_name|
+---------------+---------+
|     2026-08-07|   Friday|
|     2026-08-03|   Monday|
|     2026-05-20|Wednesday|
|     2026-01-15| Thursday|
|     2026-09-21|   Monday|
+---------------+---------+
only showing top 5 rows


In [202]:
# 7. Create year_month .
df = df.withColumn("year_month", date_format("production_date", "yyyy-MM"))
df.select("production_date", "year_month").show(5)

+---------------+----------+
|production_date|year_month|
+---------------+----------+
|     2026-08-07|   2026-08|
|     2026-08-03|   2026-08|
|     2026-05-20|   2026-05|
|     2026-01-15|   2026-01|
|     2026-09-21|   2026-09|
+---------------+----------+
only showing top 5 rows


In [203]:
# 8. Calculate monthly production.
df.groupBy("year_month").agg(sum("units_produced").alias("monthly_production")).orderBy("year_month").show()

+----------+------------------+
|year_month|monthly_production|
+----------+------------------+
|   2026-01|           8478059|
|   2026-02|           7810891|
|   2026-03|           8441121|
|   2026-04|           8211005|
|   2026-05|           8507542|
|   2026-06|           8400505|
|   2026-07|           8476416|
|   2026-08|           8529150|
|   2026-09|           8115221|
|   2026-10|           5957978|
+----------+------------------+



In [204]:
# 9. Calculate monthly defects.
df.groupBy("year_month").agg(sum("defective_units").alias("monthly_defects")).orderBy("year_month").show()

+----------+---------------+
|year_month|monthly_defects|
+----------+---------------+
|   2026-01|         211741|
|   2026-02|         197462|
|   2026-03|         212036|
|   2026-04|         208707|
|   2026-05|         212049|
|   2026-06|         210381|
|   2026-07|         210482|
|   2026-08|         214807|
|   2026-09|         202990|
|   2026-10|         149589|
+----------+---------------+



In [205]:
# 10. Calculate production by month and plant.
df.groupBy("year_month", "plant_id").agg(sum("units_produced").alias("total_units")).orderBy("year_month", "plant_id").show()

+----------+--------+-----------+
|year_month|plant_id|total_units|
+----------+--------+-----------+
|   2026-01|    PL01|    1451436|
|   2026-01|    PL02|    1419252|
|   2026-01|    PL03|    1429811|
|   2026-01|    PL04|    1399488|
|   2026-01|    PL05|    1375887|
|   2026-01|    PL06|    1402185|
|   2026-02|    PL01|    1342273|
|   2026-02|    PL02|    1278581|
|   2026-02|    PL03|    1305531|
|   2026-02|    PL04|    1292557|
|   2026-02|    PL05|    1324271|
|   2026-02|    PL06|    1267678|
|   2026-03|    PL01|    1442140|
|   2026-03|    PL02|    1359740|
|   2026-03|    PL03|    1416484|
|   2026-03|    PL04|    1447603|
|   2026-03|    PL05|    1411435|
|   2026-03|    PL06|    1363719|
|   2026-04|    PL01|    1350450|
|   2026-04|    PL02|    1369684|
+----------+--------+-----------+
only showing top 20 rows


In [206]:
# 11. Find production between two dates.
df.filter(col("production_date").between("2023-01-01", "2023-03-31")).show(5)

+-------------+---------------+--------+----------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|year_month|
+-------------+---------------+--------+----------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----------+
+-------------+---------------+--------+----------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+-------

In [207]:
# 12. Add 7 days to production date.
df.select("production_date", date_add("production_date", 7).alias("plus_7_days")).show(5)

+---------------+-----------+
|production_date|plus_7_days|
+---------------+-----------+
|     2026-08-07| 2026-08-14|
|     2026-08-03| 2026-08-10|
|     2026-05-20| 2026-05-27|
|     2026-01-15| 2026-01-22|
|     2026-09-21| 2026-09-28|
+---------------+-----------+
only showing top 5 rows


In [208]:
# 13. Subtract 30 days.
df.select("production_date", date_sub("production_date", 30).alias("minus_30_days")).show(5)

+---------------+-------------+
|production_date|minus_30_days|
+---------------+-------------+
|     2026-08-07|   2026-07-08|
|     2026-08-03|   2026-07-04|
|     2026-05-20|   2026-04-20|
|     2026-01-15|   2025-12-16|
|     2026-09-21|   2026-08-22|
+---------------+-------------+
only showing top 5 rows


In [209]:
# 14. Calculate difference from current date.
df.select("production_date", datediff(current_date(), col("production_date")).alias("days_ago")).show(5)

+---------------+--------+
|production_date|days_ago|
+---------------+--------+
|     2026-08-07|      77|
|     2026-08-03|      81|
|     2026-05-20|     156|
|     2026-01-15|     281|
|     2026-09-21|      32|
+---------------+--------+
only showing top 5 rows


In [210]:
# 15. Find the month with the highest total output.
df.groupBy("year_month").agg(sum("units_produced").alias("total_output")).orderBy(desc("total_output")).limit(1).show()

+----------+------------+
|year_month|total_output|
+----------+------------+
|   2026-08|     8529150|
+----------+------------+

